In [1]:
# importar librerias
import json
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


In [2]:
# rutas base
REPO_ROOT = Path('.').resolve()
RUN_ID = datetime.today().strftime('%Y%m%d')
INPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '01_lectura' / RUN_ID
OUTPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '02_train_test' / RUN_ID
REPORTS_DIR = REPO_ROOT / 'reports' / '02_train_test' / RUN_ID

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

input_file = INPUT_DIR / 'df_triage_encoded.parquet'
if not input_file.exists():
    candidates = sorted((REPO_ROOT / 'data' / 'intermediate' / '01_lectura').glob('*/df_triage_encoded.parquet'))
    if candidates:
        input_file = candidates[-1]
        print(f'Usando ultimo parquet encontrado: {input_file}')
    else:
        raise FileNotFoundError('No se encontro df_triage_encoded.parquet en data/intermediate/01_lectura')

print('Rutas configuradas:')
print(f'INPUT_FILE: {input_file}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'REPORTS_DIR: {REPORTS_DIR}')


Rutas configuradas:
INPUT_FILE: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\01_lectura\20260111\df_triage_encoded.parquet
OUTPUT_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\02_train_test\20260111
REPORTS_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\02_train_test\20260111


In [3]:
df = pd.read_parquet(input_file)

X = df.drop(columns=['nivel_triage'])
y = df['nivel_triage']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [4]:
# guardar archivos
X_train.to_parquet(OUTPUT_DIR / 'X_train.parquet')
X_test.to_parquet(OUTPUT_DIR / 'X_test.parquet')
y_train.to_frame().to_parquet(OUTPUT_DIR / 'y_train.parquet')
y_test.to_frame().to_parquet(OUTPUT_DIR / 'y_test.parquet')

# calcular metricas descriptivas
total_samples = len(df)
n_features = X.shape[1]

class_dist_total = y.value_counts(normalize=True).mul(100).round(2).to_dict()
class_dist_train = y_train.value_counts(normalize=True).mul(100).round(2).to_dict()
class_dist_test = y_test.value_counts(normalize=True).mul(100).round(2).to_dict()

metrics = {
    'total_samples': total_samples,
    'n_features': n_features,
    'class_distribution_percent': class_dist_total,
    'train_distribution_percent': class_dist_train,
    'test_distribution_percent': class_dist_test,
}

# guardar metricas en json
with open(REPORTS_DIR / 'split_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=4)

# mostrar en consola
print('Metricas del split:')
print(f'Total de muestras: {total_samples}')
print(f'Total de features: {n_features}')
print('Distribucion de clases (%):')
print('  Clase | Total | Train | Test')
print('  ------|-------|-------|------')
for c in sorted(y.unique()):
    total = class_dist_total.get(c, 0)
    train = class_dist_train.get(c, 0)
    test = class_dist_test.get(c, 0)
    print(f'   {c:<5} | {total:>5.1f}% | {train:>5.1f}% | {test:>5.1f}%')

print(f"Metricas guardadas en: {REPORTS_DIR / 'split_metrics.json'}")


Metricas del split:
Total de muestras: 560486
Total de features: 545
Distribucion de clases (%):
  Clase | Total | Train | Test
  ------|-------|-------|------
   1     |   9.3% |   9.3% |   9.3%
   2     |  19.4% |  19.4% |  19.4%
   3     |  21.3% |  21.3% |  21.3%
   4     |  22.3% |  22.3% |  22.3%
   5     |  27.7% |  27.7% |  27.7%
Metricas guardadas en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\02_train_test\20260111\split_metrics.json
